# StayNest - Session 6 Assignment (PySpark Deep Dive)
Work through the 8 tasks below in order. Read the Assignment Questions PDF for the
full detail and acceptance criteria. Fill in each `# TODO` cell, run it, and keep the
output visible. Run on Databricks Free Edition (serverless).

## Section 0 - Setup (already done for you)
Upload `bookings.csv`, `hotels.csv`, `customers.csv` to a Volume, then set `BASE`
to that path and run this cell. Counts should be 12000 / 200 / 2000.

In [0]:
# Point BASE at YOUR Volume path
BASE = "/Volumes/workspace/default/staynest"

print(spark.version)

read_csv = lambda name: (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{BASE}/{name}.csv"))

bookings_df   = read_csv("bookings")
hotels_df     = read_csv("hotels")
customers_df  = read_csv("customers")

print(f"bookings: {bookings_df.count()}, "
      f"hotels: {hotels_df.count()}, "
      f"customers: {customers_df.count()}")

4.1.0
bookings: 12000, hotels: 200, customers: 2000


## Task 1 - Read and inspect
Show the schema, a few sample rows, the row count, and summary stats for the
numeric columns of `bookings_df`.

In [0]:
# TODO: printSchema, show(5), count, describe(...)

bookings_df.printSchema()

bookings_df.show(5)

bookings_df.count()

bookings_df.describe().show()


root
 |-- booking_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- hotel_id: integer (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- nights: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)

+----------+-----------+--------+------------+---------+------+--------+---------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|
+----------+-----------+--------+------------+---------+------+--------+---------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|
+--------

## Task 2 - Select and filter
From `bookings_df`, select a few useful columns and return the **completed**
bookings with `amount` over 10000 in the cities Goa or Mumbai. Use `col()`, combine
conditions with `&`, and use `.isin(...)`.

In [0]:
# TODO
from pyspark.sql.functions import col
completed_bookings_Goa_Mumbai = bookings_df.filter(
    (col("status") == "completed") & (col("amount") > 10000) & (col("city").isin("Goa","Mumbai"))
    ) .select(col("booking_id"),col("hotel_id"),col("city"),col("amount"),col("status"))
                                  
completed_bookings_Goa_Mumbai.show()
                                        

+----------+--------+------+--------+---------+
|booking_id|hotel_id|  city|  amount|   status|
+----------+--------+------+--------+---------+
|   9000007|    3127|Mumbai|15693.64|completed|
|   9000010|    3054|   Goa|30786.36|completed|
|   9000028|    3022|   Goa|27008.87|completed|
|   9000034|    3118|   Goa|64388.77|completed|
|   9000050|    3147|Mumbai|24275.14|completed|
|   9000061|    3115|Mumbai|19180.93|completed|
|   9000070|    3118|   Goa|58790.91|completed|
|   9000071|    3159|Mumbai|47548.35|completed|
|   9000075|    3006|   Goa|18043.52|completed|
|   9000087|    3099|   Goa|21529.88|completed|
|   9000090|    3125|   Goa|10372.32|completed|
|   9000094|    3184|   Goa|38954.16|completed|
|   9000096|    3099|   Goa|18394.79|completed|
|   9000110|    3148|Mumbai|69186.05|completed|
|   9000113|    3006|   Goa|18727.87|completed|
|   9000119|    3022|   Goa|25395.82|completed|
|   9000122|    3166|   Goa|27690.95|completed|
|   9000123|    3099|   Goa| 22120.1|com

## Task 3 - Derived columns
Add: `amount_with_gst` (amount plus 12% tax), a `value_tier`
(premium / standard / budget) using `when`/`otherwise`, and a `booking_month`
from `booking_date`.

In [0]:
# TODO
from pyspark.sql.functions import col,round,when,month

bookings_additional_columns = bookings_df.withColumn("amount_with_gst",(round(( col("amount") + col("amount") * 12/100),2)))\
                                .withColumn("value_tier",
                                            when(col("amount") > 10000, "premium")
                                            .when(col("amount") > 5000,"standard")
                                            .otherwise("budget")) \
                                .withColumn("booking_month", month(col("booking_date")))

bookings_additional_columns.show(5)


+----------+-----------+--------+------------+---------+------+--------+---------+---------------+----------+-------------+
|booking_id|customer_id|hotel_id|booking_date|     city|nights|  amount|   status|amount_with_gst|value_tier|booking_month|
+----------+-----------+--------+------------+---------+------+--------+---------+---------------+----------+-------------+
|   9000000|     701600|    3095|  2025-11-27|   Jaipur|     4| 6087.65|completed|        6818.17|  standard|           11|
|   9000001|     700065|    3057|  2025-11-06|    Delhi|     1| 8211.19|cancelled|        9196.53|  standard|           11|
|   9000002|     701392|    3187|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|         8037.7|  standard|            8|
|   9000003|     700867|    3112|  2025-03-22|Bengaluru|     5| 7880.62|completed|        8826.29|  standard|            3|
|   9000004|     701521|    3043|  2025-04-19|   Mumbai|     5|21021.51|  pending|       23544.09|   premium|            4|
+-------

## Task 4 - Aggregations
For **completed** bookings, group by `city` and return: number of bookings, total
revenue, average amount, biggest booking, and the count of unique customers.
Order by revenue, highest first.

In [0]:
# TODO
from pyspark.sql.functions import count,avg,sum,max, countDistinct

bookings_aggregation = bookings_additional_columns.filter(col("status")=="completed")\
                                  .groupby(col("city"))\
                                  .agg(
                                      count(col("booking_id")).alias("number_of_bookings"),
                                      round(sum(col("amount")),2).alias("total_revenue") ,
                                      round(avg(col("amount")),2).alias("average_amount") ,
                                      max(col("amount")).alias("biggest_booking") ,
                                      countDistinct(col("customer_id")).alias("unique_cust_count")  )\
                                  .orderBy(col("total_revenue").desc())

bookings_aggregation.display()
                                  


city,number_of_bookings,total_revenue,average_amount,biggest_booking,unique_cust_count
Goa,2546,4.459670179E7,17516.38,78481.24,1441
Mumbai,1715,3.624122112E7,21131.91,78524.52,1153
Delhi,1174,2.631428154E7,22414.21,78669.69,861
Jaipur,979,2.443685313E7,24961.03,76904.87,796
Bengaluru,1318,2.267013697E7,17200.41,78568.81,969
Udaipur,691,1.209442742E7,17502.79,77939.5,592
Rishikesh,407,8606121.58,21145.26,77458.26,363
Manali,480,6235480.68,12990.58,77291.17,409
Munnar,244,3979216.11,16308.26,65177.15,233
Anantapur,106,2257080.65,21293.21,78237.19,105


## Task 5 - Joins
Inner-join bookings to hotels to enrich each booking. Do a left join too. Use
`left_anti` to check for orphaned bookings (expect 0). Then do a three-way join
with customers.

In [0]:
# TODO

booking_hotels_inner = bookings_df.join(hotels_df, on ="hotel_id" ,how ="inner")

booking_hotels_inner.show()

booking_hotels_left = bookings_df.join(hotels_df, on ="hotel_id" ,how ="left")

booking_hotels_left.show()

orphaned_bookings = bookings_df.join( hotels_df, on ="hotel_id" ,how ="left_anti")

orphaned_bookings.show()

booking_hotels_customer =  bookings_df.join(hotels_df, on ="hotel_id" ,how ="inner") \
                                      .join(customers_df, on ="customer_id" ,how ="inner")
                    
booking_hotels_customer.show()



+--------+----------+-----------+------------+---------+------+--------+---------+----------------+---------+--------+-----------+
|hotel_id|booking_id|customer_id|booking_date|     city|nights|  amount|   status|      hotel_name|     city|category|star_rating|
+--------+----------+-----------+------------+---------+------+--------+---------+----------------+---------+--------+-----------+
|    3095|   9000000|     701600|  2025-11-27|   Jaipur|     4| 6087.65|completed|   Orchid Suites|   Jaipur|  Budget|        3.8|
|    3057|   9000001|     700065|  2025-11-06|    Delhi|     1| 8211.19|cancelled|  Orchid Retreat|    Delhi|  Luxury|        4.1|
|    3187|   9000002|     701392|  2025-08-21|   Jaipur|     2| 7176.52|cancelled|   Marigold Stay|   Jaipur| Premium|        3.8|
|    3112|   9000003|     700867|  2025-03-22|Bengaluru|     5| 7880.62|completed|      Grand Stay|Bengaluru|  Budget|        3.6|
|    3043|   9000004|     701521|  2025-04-19|   Mumbai|     5|21021.51|  pending| 

## Task 6 - Spark SQL + a window function
Register temp views and use `spark.sql` to get revenue by hotel `category` for
completed bookings. Then use a window function to rank the **top 3 hotels by
revenue within each city**.

In [0]:
# TODO: spark.sql(...) for revenue by category

bookings_df.createOrReplaceTempView("bookings")
customers_df.createOrReplaceTempView("customers")
hotels_df.createOrReplaceTempView("hotels")

Revenue_byhotelcategory =spark.sql("""
select h.category as hotel_category, round(sum(amount),2) as revenue_hotel from bookings b
join hotels h on b.hotel_id = h.hotel_id
where b.status ="completed"
group by h.category


""")

Revenue_byhotelcategory.display()

hotel_category,revenue_hotel
Premium,5.825564086E7
Budget,2.233825323E7
Luxury,1.068376269E8


In [0]:
# TODO: window function for top 3 hotels per city

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number


Hotel_Revenue_bycity =spark.sql("""
select b.city,h.hotel_name , round(sum(amount),2) as revenue_hotel from bookings b
join hotels h on b.hotel_id = h.hotel_id
where b.status ="completed"
group by b.city,h.hotel_name 
order by b.city


""")

#Hotel_Revenue_bycity.show()

w = Window.partitionBy("city").orderBy(col("revenue_hotel").desc())

top3_hotels_by_revenue_city = (Hotel_Revenue_bycity
                               .withColumn("rank_in_city", row_number().over(w))
                               .filter(col("rank_in_city") <=3 )

)
top3_hotels_by_revenue_city.show()

+---------+----------------+-------------+------------+
|     city|      hotel_name|revenue_hotel|rank_in_city|
+---------+----------------+-------------+------------+
|Anantapur|   Azure Retreat|   1434328.39|           1|
|Anantapur|   Lotus Retreat|    617708.27|           2|
|Anantapur|Orchid Residency|    205043.99|           3|
|Bengaluru|      Serene Inn|   1867479.69|           1|
|Bengaluru|   Willow Suites|   1433492.08|           2|
|Bengaluru|   Heritage Stay|   1400383.85|           3|
|    Delhi|  Orchid Retreat|   2616612.56|           1|
|    Delhi|   Marigold Stay|   2327675.63|           2|
|    Delhi|Orchid Residency|   2264974.65|           3|
|      Goa| Cedar Residency|   5017409.03|           1|
|      Goa|      Amber Stay|   4217831.66|           2|
|      Goa|     Sunset Stay|   3626645.37|           3|
|   Jaipur|    Emerald Stay|   2686993.34|           1|
|   Jaipur| Amber Residency|    2492240.4|           2|
|   Jaipur|Orchid Residency|   2149131.67|      

## Task 7 - Write the result
Write your city-revenue result as **Parquet**, and also as a **Delta table** with
`saveAsTable`. Read the Delta table back to confirm.

In [0]:
# TODO

top3_hotels_by_revenue_city.write.mode("overwrite").parquet(f"{BASE}/output/top3_hotels_by_revenue_city_parquet")
print(f"Written to {BASE}/output/top3_hotels_by_revenue_city_parquet")

(top3_hotels_by_revenue_city.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("workspace.default.city_hotelrevenue"))


Written to /Volumes/workspace/default/staynest/output/top3_hotels_by_revenue_city_parquet


## Task 8 - One chained pipeline
In a single chain: keep completed bookings, join hotels, keep hotels with
`star_rating >= 4.0`, group by `category`, sum revenue, order descending, take the
top 5. End with one `.show()`.

In [0]:
# TODO

top5_hotels = ( bookings_df
               .filter(col("status")=="completed")
               .join(hotels_df ,on = "hotel_id" ,how ="inner")
               .filter(col("star_rating") >= 4.0)
               .groupby(col("category"))
               .agg(sum("amount").alias("revenue"))
               .orderBy(col("revenue").desc())
               .limit(5)

)

top5_hotels.show()



+--------+--------------------+
|category|             revenue|
+--------+--------------------+
|  Luxury|5.8701580909999914E7|
| Premium|2.2516532060000006E7|
|  Budget|   7790394.569999994|
+--------+--------------------+

